In [ ]:

import os
import requests
from datetime import datetime
from bs4 import BeautifulSoup

from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_community.tools.tavily_search import TavilySearchResults

print('✅ Done!')

✅ Done!


In [8]:

os.environ['GROQ_API_KEY']   = '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'    
os.environ['TAVILY_API_KEY'] = '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'  

USER_NAME = 'Abd'
USER_CPU  = 'Intel i3-1220P'
USER_GPU  = 'Intel Iris Xe (Integrated)'
USER_RAM  = '8GB DDR4'
BUDGET    = 'Free or Low Cost'

print(f'✅ User   : {USER_NAME}')
print(f'✅ CPU    : {USER_CPU}')
print(f'✅ GPU    : {USER_GPU}')
print(f'✅ RAM    : {USER_RAM}')
print(f'✅ Budget : {BUDGET}')

✅ User   : Abd
✅ CPU    : Intel i3-1220P
✅ GPU    : Intel Iris Xe (Integrated)
✅ RAM    : 8GB DDR4
✅ Budget : Free or Low Cost


In [11]:
# ✅ CELL 4 — All 4 Tools

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TOOL 1 — Web Search
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
search_tool = TavilySearchResults(max_results=3, name='web_search')

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TOOL 2 — Web Scraper
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@tool
def web_scraper(url: str) -> str:
    """Scrape and extract content from any game website or URL."""
    try:
        headers  = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get(url, headers=headers, timeout=10)
        soup     = BeautifulSoup(response.content, 'html.parser')
        for tag in soup(['script', 'style', 'nav', 'footer']):
            tag.decompose()
        text  = soup.get_text(separator='\n', strip=True)
        lines = [l.strip() for l in text.splitlines() if len(l.strip()) > 30]
        return '\n'.join(lines[:60])
    except Exception as e:
        return f'❌ Scraping failed: {str(e)}'

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TOOL 3 — PC Spec Checker
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@tool
def check_pc_specs(game_name: str) -> str:
    """Check whether the user's PC can run a specific game based on their hardware specs."""
    heavy_games = ['cyberpunk', 'rdr2', 'elden ring', 'warzone', 'battlefield 2042', 'ms flight simulator']
    light_games = ['valorant', 'minecraft', 'cs2', 'rocket league', 'among us', 'terraria', 'gta v', 'fortnite']

    game_lower = game_name.lower()

    if any(g in game_lower for g in heavy_games):
        return (
            f'❌ {game_name} — NOT Recommended\n'
            f'   Reason : Needs dedicated GPU\n'
            f'   Your GPU: {USER_GPU} is not powerful enough'
        )
    elif any(g in game_lower for g in light_games):
        return (
            f'✅ {game_name} — Can Run Smoothly!\n'
            f'   CPU : {USER_CPU} ✅\n'
            f'   RAM : {USER_RAM} ✅\n'
            f'   GPU : {USER_GPU} ✅'
        )
    else:
        return (
            f'⚠️ {game_name} — May Run on Low Settings\n'
            f'   CPU : {USER_CPU}\n'
            f'   RAM : {USER_RAM}\n'
            f'   GPU : {USER_GPU} (Integrated — expect 30-40 FPS)'
        )

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# TOOL 4 — Date & Time
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
@tool
def get_datetime(dummy: str = 'now') -> str:
    """Get today's date and time. Useful for finding latest or recent game releases."""
    now = datetime.now()
    return f"📅 Today: {now.strftime('%A, %d %B %Y')} | 🕐 {now.strftime('%I:%M %p')}"

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# All Tools List
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
all_tools = [search_tool, web_scraper, check_pc_specs, get_datetime]
print('✅ 4 Tools Ready!')
print('   1. web_search')
print('   2. web_scraper')
print('   3. check_pc_specs')
print('   4. get_datetime')

✅ 4 Tools Ready!
   1. web_search
   2. web_scraper
   3. check_pc_specs
   4. get_datetime


C:\Users\abhis\AppData\Local\Temp\ipykernel_15728\705543351.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results=3, name='web_search')


In [12]:

today = datetime.now().strftime('%A, %d %B %Y')

system_prompt = f"""You are GameBot 🎮 — an expert AI Game Recommender.

👤 User    : {USER_NAME}
💻 CPU     : {USER_CPU}
🎮 GPU     : {USER_GPU}
🧠 RAM     : {USER_RAM}
💰 Budget  : {BUDGET}
📅 Date    : {today}

Your job is to recommend games that run well on {USER_NAME}'s PC.
Always check PC specs before recommending.
Prefer free or cheap games. Format answers clearly with emojis."""

print('✅ System Prompt Ready!')
print('─'*40)
print(system_prompt)

✅ System Prompt Ready!
────────────────────────────────────────
You are GameBot 🎮 — an expert AI Game Recommender.

👤 User    : Abd
💻 CPU     : Intel i3-1220P
🎮 GPU     : Intel Iris Xe (Integrated)
🧠 RAM     : 8GB DDR4
💰 Budget  : Free or Low Cost
📅 Date    : Saturday, 27 June 2026

Your job is to recommend games that run well on Abd's PC.
Always check PC specs before recommending.
Prefer free or cheap games. Format answers clearly with emojis.


In [28]:
# ✅ CELL 6 — LLM + Agent (bind_tools wala — no imports needed)
llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=os.environ['GROQ_API_KEY'],
    temperature=0
)

# Bind tools to LLM
llm_with_tools = llm.bind_tools(all_tools)
tools_map = {t.name: t for t in all_tools}

print('✅ GameBot Ready! 🎮')

✅ GameBot Ready! 🎮


In [24]:
# ✅ CELL 7 — Run Agent
def run_agent(query):
    print(f"\n{'═'*50}")
    print(f'  ❓ {query}')
    print(f"{'─'*50}\n")

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=query)
    ]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        # Koi tool call nahi — final answer
        if not response.tool_calls:
            break

        # Tools run karo
        for tc in response.tool_calls:
            tool_result = tools_map[tc['name']].invoke(tc['args'])
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tc['id']))

    print('✅ Answer:\n')
    print(response.content)
    print(f"{'═'*50}")
    return response.content

In [29]:
# 🧪 TEST 1 — FPS Game Recommendation
run_agent('Recommend me best FPS games like COD that can run on my PC')


══════════════════════════════════════════════════
  ❓ Recommend me best FPS games like COD that can run on my PC
──────────────────────────────────────────────────

✅ Answer:

🎮 Based on your PC specs, here are some free or low-cost FPS games that you can play:
1. **Team Fortress 2** 🤖: A team-based FPS game with cartoonish graphics, which should run smoothly on your PC.
2. **Warframe** 🚀: A cooperative FPS game with sci-fi elements, which has low system requirements and is free to play.
3. **Counter-Strike: Source** 🏰: A classic FPS game that is relatively lightweight and should run well on your PC.
4. **Insurgency: Sandstorm** 🏜️: A tactical FPS game that is free to play and has low system requirements.
5. **Red Orchestra** 🔴: A WWII-themed FPS game that is free to play and has moderate system requirements.

These games should provide a similar experience to Call of Duty, but with lower system requirements. However, keep in mind that the performance may vary depending on the specif

"🎮 Based on your PC specs, here are some free or low-cost FPS games that you can play:\n1. **Team Fortress 2** 🤖: A team-based FPS game with cartoonish graphics, which should run smoothly on your PC.\n2. **Warframe** 🚀: A cooperative FPS game with sci-fi elements, which has low system requirements and is free to play.\n3. **Counter-Strike: Source** 🏰: A classic FPS game that is relatively lightweight and should run well on your PC.\n4. **Insurgency: Sandstorm** 🏜️: A tactical FPS game that is free to play and has low system requirements.\n5. **Red Orchestra** 🔴: A WWII-themed FPS game that is free to play and has moderate system requirements.\n\nThese games should provide a similar experience to Call of Duty, but with lower system requirements. However, keep in mind that the performance may vary depending on the specific game and your PC's configuration. 🎮"

In [30]:
# 🧪 TEST 2 — Spec Check
run_agent('Can my PC run Valorant?')


══════════════════════════════════════════════════
  ❓ Can my PC run Valorant?
──────────────────────────────────────────────────

✅ Answer:

Valorant's system requirements are:
   CPU: Intel i3-4150 or AMD equivalent
   RAM: 4GB
   GPU: Intel HD 3000 or AMD equivalent

Your PC exceeds these requirements, so you can run Valorant smoothly. 🎮💻
══════════════════════════════════════════════════


"Valorant's system requirements are:\n   CPU: Intel i3-4150 or AMD equivalent\n   RAM: 4GB\n   GPU: Intel HD 3000 or AMD equivalent\n\nYour PC exceeds these requirements, so you can run Valorant smoothly. 🎮💻"